# Imports

In [1]:
import cv2
from ultralytics import YOLO
import os
from tqdm.notebook import tqdm
import cv2
import math

# Labelling video

In [48]:
video_path_out = '{}_out.mp4'.format('video')

cap = cv2.VideoCapture('video.mp4')
ret, frame = cap.read()
H, W, _ = frame.shape
out = cv2.VideoWriter(video_path_out, cv2.VideoWriter_fourcc(*'MP4V'), int(cap.get(cv2.CAP_PROP_FPS)), (W, H))

model_path = os.path.join('.', 'runs', 'detect', 'train', 'weights', 'last.pt')

# Load a model
model = YOLO(model_path)  # load a custom model

threshold = 0.5

class_name_dict = {0: 'bee', 1:'ladybug'}


while ret:

    results = model(frame)[0]

    for result in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = result

        if score > threshold:
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 4)
            cv2.putText(frame, class_name_dict[int(class_id)].upper(), (int(x1), int(y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3, cv2.LINE_AA)

    out.write(frame)
    ret, frame = cap.read()

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 ladybug, 47.7ms
Speed: 1.0ms preprocess, 47.7ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 46.6ms
Speed: 1.0ms preprocess, 46.6ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 46.2ms
Speed: 1.0ms preprocess, 46.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 45.4ms
Speed: 2.0ms preprocess, 45.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 46.0ms
Speed: 2.0ms preprocess, 46.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 47.0ms
Speed: 2.0ms preprocess, 47.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 46.4ms
Speed: 2.0ms preprocess, 46.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 46.1ms
Speed: 2.0ms preprocess, 46.1ms inference, 0.0ms postprocess per image at shape (

# Counting

In [2]:
# Load video and prepare output video
video_path_out = '{}_out.mp4'.format('video')
cap = cv2.VideoCapture('video.mp4')
video_not_finished, frame = cap.read()
H, W, _ = frame.shape
out = cv2.VideoWriter(video_path_out, cv2.VideoWriter_fourcc(*'MP4V'), int(cap.get(cv2.CAP_PROP_FPS)), (W, H))

# Load a model
model_path = os.path.join('.', 'runs', 'detect', 'train', 'weights', 'last.pt')
model = YOLO(model_path)  # load a custom model

# Threshold
pred_threshold = 0.90
gone_insect_threshold = 10
distance_threshold = 30

# Classes
class_name_dict = {0: 'bee', 1: 'ladybug'}
bee_count = 0
ladybug_count = 0

# Dictionary to store tracked objects
tracked_insects = {}
insect_id = 0
frame_count = 0


while video_not_finished:
    results = model(frame)[0]

    # Reset the set of detected object IDs for each frame
    insect_ids = set()

    for detection in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = detection

        if score > pred_threshold:
            centroid_x = (x1 + x2) / 2
            centroid_y = (y1 + y2) / 2
            centroid = (centroid_x, centroid_y)

            class_name = class_name_dict.get(int(class_id))

            # Search for existing objects to associate with
            matched_insect_id = None
            for id, insect_data in tracked_insects.items():
                last_centroid, last_class_name, _ = insect_data
                distance = math.sqrt((last_centroid[0] - centroid_x) ** 2 + (last_centroid[1] - centroid_y) ** 2)
                if distance < distance_threshold and last_class_name == class_name:
                    matched_insect_id = id
                    break

    
            if matched_insect_id is not None:
                # Update existing object
                tracked_insects[matched_insect_id] = (centroid, class_name, frame_count)
                insect_ids.add(matched_insect_id)
            else:
                # Create new insect instance
                tracked_insects[insect_id] = (centroid, class_name, frame_count)
                insect_ids.add(insect_id)
                insect_id += 1
                if class_name=='bee':
                        bee_count+= 1
                elif class_name=='ladybug':
                        ladybug_count+= 1
            

            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 4)
            cv2.putText(frame, class_name.upper(), (int(x1), int(y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3, cv2.LINE_AA)

    # Remove gone insects
    gone_insects = []
    for id, insect_data in tracked_insects.items():
        if id not in insect_ids:
            _, _, last_frame_count = insect_data
            if frame_count - last_frame_count > gone_insect_threshold:
                gone_insects.append(id)

    for gone_insect_id in gone_insects:
        del tracked_insects[gone_insect_id]

    out.write(frame)
    video_not_finished, frame = cap.read()
    frame_count += 1
    

cap.release()
out.release()
cv2.destroyAllWindows()

print("Bee count:", bee_count)
print("Ladybug count:", ladybug_count)



0: 384x640 1 ladybug, 96.0ms
Speed: 4.0ms preprocess, 96.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 57.0ms
Speed: 3.0ms preprocess, 57.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 58.0ms
Speed: 2.0ms preprocess, 58.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 56.0ms
Speed: 2.0ms preprocess, 56.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 54.0ms
Speed: 1.0ms preprocess, 54.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 50.4ms
Speed: 2.0ms preprocess, 50.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 51.0ms
Speed: 1.0ms preprocess, 51.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 384x640 1 ladybug, 57.0ms
Speed: 1.0ms preprocess, 57.0ms inference, 0.0ms postprocess per image at shape (

Bee count: 0
Ladybug count: 11
